# From PyTorch to NengoDL

This tutorial is aimed at PyTorch users who want to understand NengoDL. It mirrors the structure of the original [NengoDL tutorial](https://www.nengo.ai/nengo-dl/examples/from-tensorflow.html), replacing TensorFlow concepts with PyTorch equivalents.

**NengoDL** bridges Nengo's biologically-inspired neural simulation framework with PyTorch's automatic differentiation and GPU support. Key capabilities:

- Build networks with spiking or rate neurons using Nengo's population API
- Train them end-to-end with backpropagation via PyTorch optimizers
- Embed arbitrary `nn.Module` layers inside Nengo simulations (`TorchNode`)
- Deploy identical network definitions to neuromorphic hardware

**Sections:**
1. What is Nengo
2. Simulating a Network
3. Spiking Networks
4. Inserting PyTorch Code
5. Deep Learning Parameter Optimization
6. NEF Parameter Optimization
7. Running on Neuromorphic Hardware
8. Half-Domain Generalization

In [ ]:
%matplotlib inline
import warnings
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import nengo
import nengo_dl
warnings.simplefilter('ignore')

seed = 0
np.random.seed(seed)
torch.manual_seed(seed)

n_in = 784      # MNIST image size (28x28)
n_hidden = 64   # bottleneck / hidden layer size

## 1. What is Nengo

Nengo is a Python library for building and simulating neural networks in a way that reflects how the brain works. It was not originally designed for deep learning — it comes from computational neuroscience — but NengoDL extends it with full gradient-based training.

The core difference from PyTorch:

| PyTorch | Nengo |
|---------|-------|
| `nn.Linear` + `F.relu` | `nengo.Ensemble` with `neuron_type=nengo.RectifiedLinear()` |
| `nn.Linear` weight matrix | `nengo.Connection` with a `transform` |
| Tensor output of a layer | `nengo.Probe` on an ensemble |
| `model.forward(x)` | `sim.run_steps(n, data={node: x})` |

Below we build the same shallow autoencoder in both frameworks.

In [ ]:
# ── PyTorch autoencoder ────────────────────────────────────────────────
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Linear(n_in, n_hidden)
        self.decoder = nn.Linear(n_hidden, n_in)

    def forward(self, x):
        x = F.relu(self.encoder(x))
        return F.relu(self.decoder(x))

pt_model = Autoencoder()
print('PyTorch model:')
print(pt_model)

In [ ]:
# ── Equivalent Nengo network ────────────────────────────────────────────
# nengo.Ensemble corresponds to a layer of neurons.
# nengo.Connection with transform is the weight matrix.
# nengo.Probe records output (like capturing a tensor).
with nengo.Network(seed=seed) as auto_net:
    inp_auto = nengo.Node(np.zeros(n_in))

    # Encoder layer: n_in → n_hidden, ReLU neurons
    enc = nengo.Ensemble(n_hidden, 1, neuron_type=nengo.RectifiedLinear(), seed=0)
    nengo.Connection(inp_auto, enc.neurons,
                     transform=np.random.randn(n_hidden, n_in) * 0.01,
                     synapse=None)

    # Decoder layer: n_hidden → n_in, ReLU neurons
    dec = nengo.Ensemble(n_in, 1, neuron_type=nengo.RectifiedLinear(), seed=1)
    nengo.Connection(enc.neurons, dec.neurons,
                     transform=np.random.randn(n_in, n_hidden) * 0.01,
                     synapse=None)

    p_auto = nengo.Probe(dec.neurons, synapse=None)

print('Nengo autoencoder:')
print(f'  Input node:       {n_in} dims')
print(f'  Encoder ensemble: {n_hidden} neurons  ← nn.Linear({n_in}, {n_hidden}) + ReLU')
print(f'  Decoder ensemble: {n_in} neurons  ← nn.Linear({n_hidden}, {n_in}) + ReLU')
print(f'  Probe:            {p_auto}  ← captures output like a tensor')

## 2. Simulating a Network

In PyTorch you call `model(x)` — a simple function from inputs to outputs. In NengoDL you call `sim.run_steps(n_steps, data={node: x})`. The crucial difference is that **Nengo adds a time dimension**:

- PyTorch input: `(batch, features)`
- NengoDL input: `(batch, n_steps, features)`

This reflects that Nengo networks are inherently temporal and stateful — neurons have voltages and refractory periods that evolve over time. Even when running for a single step, the time axis must be explicit.

In [ ]:
# PyTorch: forward pass — no time dimension
x = torch.zeros(5, n_in)
with torch.no_grad():
    pt_out = pt_model(x)
print('PyTorch forward pass:')
print(f'  input:  {tuple(x.shape)}   (batch, features)')
print(f'  output: {tuple(pt_out.shape)}  (batch, features)')
print()

# NengoDL: run_steps — explicit time dimension required
# Input shape: (batch, n_steps, features)
x_nd = np.zeros((5, 1, n_in), dtype=np.float32)   # n_steps=1
with nengo_dl.Simulator(auto_net, minibatch_size=5) as sim:
    sim.run_steps(1, data={inp_auto: x_nd})
    nd_out = sim.data[p_auto]   # (5, 1, n_in) — batch kept since minibatch_size>1

print('NengoDL sim.run_steps():')
print(f'  input:  {x_nd.shape}  (batch, n_steps, features)')
print(f'  output: {nd_out.shape}  (batch, n_steps, features)')
print()
print('The n_steps=1 "time" dimension is always present.')
print('Use sim.get_data(probe) or sim.data[probe] to retrieve results.')

## 3. Spiking Networks

One of Nengo's most powerful features is the ability to switch between **rate** and **spiking** neuron models with a single line change:

- **Rate neurons** (`nengo.RectifiedLinear`, `nengo.LIFRate`): output a continuous firing rate at each timestep — analogous to standard ReLU activations. This is what most deep learning uses.

- **Spiking neurons** (`nengo.SpikingRectifiedLinear`, `nengo.LIF`): emit discrete all-or-nothing spikes. Information is encoded in the *timing* and *rate* of spikes. This is the biologically realistic model and is required for neuromorphic hardware.

A network trained with rate neurons can often be directly converted to spiking neurons with minimal accuracy loss.

In [ ]:
with nengo.Network(seed=seed) as spike_net:
    stim = nengo.Node(lambda t: np.sin(2 * np.pi * t))

    # Rate neurons — continuous, differentiable output
    b_rate = nengo.Ensemble(10, 1, neuron_type=nengo.RectifiedLinear(), seed=2)
    nengo.Connection(stim, b_rate)

    # Spiking neurons — discrete events, same tuning curves
    b_spike = nengo.Ensemble(10, 1, neuron_type=nengo.SpikingRectifiedLinear(), seed=2)
    nengo.Connection(stim, b_spike)

    p_stim  = nengo.Probe(stim)
    p_rate  = nengo.Probe(b_rate.neurons)
    p_spike = nengo.Probe(b_spike.neurons)

with nengo_dl.Simulator(spike_net) as sim:
    sim.run_steps(1000)

t = sim.trange()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(t, sim.data[p_stim])
axes[0].set_title('Input signal: sin(2πt)')
axes[0].set_xlabel('Time (s)')

axes[1].plot(t, sim.data[p_rate])
axes[1].set_title('Rate neurons (continuous output)')
axes[1].set_xlabel('Time (s)')

spike_data = sim.data[p_spike]
for i in range(spike_data.shape[1]):
    spk_t = t[spike_data[:, i] > 0]
    axes[2].vlines(spk_t, i - 0.4, i + 0.4, lw=0.8)
axes[2].set_title('Spiking neurons (discrete spike raster)')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('Neuron index')

plt.tight_layout()
plt.show()
print(f'Rate neuron max firing rate: {sim.data[p_rate].max():.1f}')

### Synaptic filtering

Raw spike trains are noisy. A **synaptic low-pass filter** (leaky integrator with time constant τ) smooths them into an estimate of the underlying firing rate. In biology this corresponds to the decay of post-synaptic potentials.

Longer τ → smoother estimate, more temporal lag. Shorter τ → more responsive, noisier. The `nengo.Lowpass` synapse can be applied to probes or connections.

In [ ]:
tau = 0.05   # 50 ms time constant
filt = nengo.Lowpass(tau)
filtered = filt.filt(sim.data[p_spike], dt=0.001)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(t, sim.data[p_rate])
axes[0].set_title('Rate neurons (ground truth)')
axes[0].set_xlabel('Time (s)')

axes[1].plot(t, filtered)
axes[1].set_title(f'Filtered spike trains (\u03c4 = {tau} s)')
axes[1].set_xlabel('Time (s)')

plt.suptitle('Synaptic filtering recovers a smooth rate estimate from spikes',
             fontsize=12)
plt.tight_layout()
plt.show()

## 4. Inserting PyTorch Code

Nengo doesn't have built-in support for operations like batch normalization or custom learned transformations. `nengo_dl.TorchNode` solves this by wrapping any `torch.nn.Module` and embedding it directly inside the Nengo simulation loop.

Key properties of `TorchNode`:
- Receives input as `(batch, n_features)` each timestep
- Gradients flow through it — parameters are trained with `sim.fit()`
- Any `nn.Module` works: `BatchNorm1d`, `LayerNorm`, attention layers, etc.

Below we apply layer normalization after a Nengo ensemble — something not possible with plain Nengo.

In [ ]:
class NormLayer(nn.Module):
    """Layer normalization — works with any batch size, unlike BatchNorm1d."""
    def __init__(self, n_features):
        super().__init__()
        self.norm = nn.LayerNorm(n_features)

    def forward(self, x):
        return self.norm(x)


with nengo.Network(seed=seed) as norm_net:
    inp_norm = nengo.Node(np.zeros(n_in))

    # Standard Nengo ensemble (encoder layer)
    ens = nengo.Ensemble(n_hidden, 1, neuron_type=nengo.RectifiedLinear(), seed=0)
    nengo.Connection(inp_norm, ens.neurons,
                     transform=np.random.randn(n_hidden, n_in) * 0.01,
                     synapse=None)

    # TorchNode: embeds a PyTorch module in the Nengo simulation
    # This is analogous to nengo_dl.TensorNode in the original NengoDL
    norm_node = nengo_dl.TorchNode(
        NormLayer(n_hidden),
        shape_in=(n_hidden,),
        shape_out=(n_hidden,)
    )
    nengo.Connection(ens.neurons, norm_node, synapse=None)

    p_raw  = nengo.Probe(ens.neurons, synapse=None)
    p_norm = nengo.Probe(norm_node, synapse=None)

x_sample = np.random.randn(1, 1, n_in).astype(np.float32)
with nengo_dl.Simulator(norm_net) as sim:
    sim.run_steps(1, data={inp_norm: x_sample})

raw   = sim.data[p_raw][0]    # (1, n_hidden) -> shape [n_hidden]
normd = sim.data[p_norm][0]
print('Before LayerNorm:')
print(f'  mean={raw.mean():.4f}  std={raw.std():.4f}  min={raw.min():.4f}  max={raw.max():.4f}')
print('After LayerNorm (via TorchNode):')
print(f'  mean={normd.mean():.4f}  std={normd.std():.4f}  min={normd.min():.4f}  max={normd.max():.4f}')
print()
print('TorchNode lets you embed any nn.Module — BatchNorm, attention,')
print('pretrained layers, custom losses — directly inside a Nengo network.')

## 5. Deep Learning Parameter Optimization

NengoDL provides a Keras-style training API backed by PyTorch:

```python
# PyTorch
model.fit(x_train, y_train, epochs=10)            # Keras-style
# or manually: optimizer.zero_grad(); loss.backward(); optimizer.step()

# NengoDL
sim.compile(optimizer='adam', loss={probe: 'mse'})
sim.fit(x={node: x_train}, y={probe: y_train}, n_steps=1, epochs=10)
```

The key difference is the **temporal dimension**: NengoDL expects inputs of shape `(n_samples, n_steps, n_features)` instead of `(n_samples, n_features)`. For a single-step "static" network, `n_steps=1`.

We demonstrate this by training a Nengo autoencoder on MNIST images.

In [ ]:
# Load MNIST data
try:
    import torchvision

    train_ds = torchvision.datasets.MNIST(root='/tmp/mnist', train=True, download=True)
    test_ds = torchvision.datasets.MNIST(root='/tmp/mnist', train=False, download=True)

    # Match the reference TensorFlow example: flatten MNIST but keep pixel
    # values on the native 0..255 scale.
    x_train = train_ds.data.numpy().reshape(-1, 784).astype(np.float32)
    x_test = test_ds.data.numpy().reshape(-1, 784).astype(np.float32)
    n_pixels = 784
    img_side = 28
    pixel_max = 255.0
    print('Loaded MNIST (28x28 = 784 pixels)')

except Exception:
    from sklearn.datasets import load_digits

    digits = load_digits()
    imgs = digits.images.reshape(-1, 64).astype(np.float32)
    split = int(0.85 * len(imgs))
    x_train, x_test = imgs[:split], imgs[split:]
    n_pixels = 64
    img_side = 8
    pixel_max = 16.0
    print('torchvision not found; using sklearn digits (8x8 = 64 pixels)')

print(f'  x_train: {x_train.shape}  x_test: {x_test.shape}')
print(f'  pixel range: [{x_train.min():.0f}, {x_train.max():.0f}]')

# Show sample images
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(x_train[i].reshape(img_side, img_side), cmap='gray', vmin=0, vmax=pixel_max)
    ax.axis('off')
plt.suptitle('Sample MNIST training images')
plt.tight_layout()
plt.show()


In [ ]:
# Build a Nengo autoencoder for images.
# This mirrors the reference TensorFlow/Keras example:
#   input -> Dense(64, relu) -> Dense(784, relu)
n_hid_ae = 64

with nengo.Network(seed=seed) as mnist_net:
    inp_mnist = nengo.Node(np.zeros(n_pixels))

    # Match the Keras Dense+ReLU layers by using neuron gains=1 and biases=0.
    hidden = nengo.Ensemble(
        n_hid_ae,
        1,
        neuron_type=nengo.RectifiedLinear(),
        gain=nengo.dists.Choice([1]),
        bias=nengo.dists.Choice([0]),
        seed=10,
    )
    nengo.Connection(
        inp_mnist,
        hidden.neurons,
        transform=nengo_dl.dists.Glorot(),
        synapse=None,
    )

    recon = nengo.Ensemble(
        n_pixels,
        1,
        neuron_type=nengo.RectifiedLinear(),
        gain=nengo.dists.Choice([1]),
        bias=nengo.dists.Choice([0]),
        seed=11,
    )
    nengo.Connection(
        hidden.neurons,
        recon.neurons,
        transform=nengo_dl.dists.Glorot(),
        synapse=None,
    )

    p_dec = nengo.Probe(recon.neurons, synapse=None)

print('Nengo autoencoder:')
print(f'  Input:   {n_pixels} dims')
print(f'  Encoder: {n_hid_ae} ReLU neurons  <- nn.Linear({n_pixels}, {n_hid_ae}) + ReLU')
print(f'  Decoder: {n_pixels} ReLU neurons  <- nn.Linear({n_hid_ae}, {n_pixels}) + ReLU')


In [ ]:
# ── Training ──────────────────────────────────────────────────────────────
# Reshape data: (n_samples, n_pixels) -> (n_samples, n_steps=1, n_pixels)
# This is the key shape difference vs PyTorch's model.fit().
x_train_nd = x_train.reshape(-1, 1, n_pixels)
x_test_nd = x_test.reshape(-1, 1, n_pixels)

n_epochs = 2
minibatch = 50
params_path = '/tmp/mnist_ae_pytorch_reference'

with nengo_dl.Simulator(mnist_net, minibatch_size=minibatch, seed=seed) as sim:
    # Match the reference TensorFlow example: RMSprop(1e-3), MSE, full train set.
    sim.compile(optimizer='rmsprop', loss={p_dec: 'mse'})

    history = sim.fit(
        x={inp_mnist: x_train_nd},
        y={p_dec: x_train_nd},
        n_steps=1,
        epochs=n_epochs,
    )
    sim.save_params(params_path)

plt.figure(figsize=(8, 4))
plt.plot(history['loss'], marker='o')
plt.title('MNIST autoencoder — training loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.tight_layout()
plt.show()


In [ ]:
# Show original vs reconstructed images
n_show = 8
with nengo_dl.Simulator(mnist_net, minibatch_size=n_show, seed=seed) as sim:
    sim.load_params(params_path)
    sim.run_steps(1, data={inp_mnist: x_test_nd[:n_show]})
    recons = sim.data[p_dec][:, 0, :]
    recons = np.clip(recons, 0, pixel_max)

fig, axes = plt.subplots(2, n_show, figsize=(12, 3))
for i in range(n_show):
    axes[0, i].imshow(
        x_test[i].reshape(img_side, img_side), cmap='gray', vmin=0, vmax=pixel_max
    )
    axes[0, i].axis('off')
    axes[1, i].imshow(
        recons[i].reshape(img_side, img_side), cmap='gray', vmin=0, vmax=pixel_max
    )
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=9)
axes[1, 0].set_ylabel('Reconstructed', fontsize=9)
plt.suptitle('Autoencoder: original (top) vs reconstructed (bottom)')
plt.tight_layout()
plt.show()

mse = np.mean((x_test[:n_show] - recons) ** 2)
print(f'Reconstruction MSE on displayed examples: {mse:.2f}')


## 6. NEF Parameter Optimization

NengoDL supports a second, completely different optimization method: the **Neural Engineering Framework (NEF)**. Instead of gradient descent, NEF finds connection weights analytically using **linear least squares**.

When you write `nengo.Connection(ens, out, function=f)`, Nengo automatically:
1. Evaluates the ensemble's tuning curves at many sample points
2. Solves the linear system `A @ d ≈ f(x)` for decoder weights `d`
3. Stores the result as the connection weights

This happens **at build time** — no `sim.fit()` needed.

Here we use the same random temporal input style as the original NengoDL example: a `WhiteSignal` drives spiking Nengo ensembles, and the output is compared against `sin(x²)` over time.

In [ ]:
# NEF automatically finds decoder weights for sin(x²) using least squares.
# No gradient descent — happens at sim build time.

n_steps_nef = 1000
dt = 0.001

with nengo.Network(seed=seed) as nef_net:
    # Random temporal input for x, matching the original NengoDL example.
    inp_nef = nengo.Node(nengo.processes.WhiteSignal(1, 5, rms=0.3, seed=seed))

    # First ensemble computes x²; second ensemble computes sin(x²).  These are
    # spiking LIF ensembles by default, so the decoded output has realistic
    # temporal variability rather than a perfectly smooth rate curve.
    ens_square = nengo.Ensemble(50, dimensions=1)
    ens_sin = nengo.Ensemble(50, dimensions=1)

    nengo.Connection(inp_nef, ens_square)
    nengo.Connection(ens_square, ens_sin, function=np.square)

    outpt_nef = nengo.Node(size_in=1)
    nengo.Connection(ens_sin, outpt_nef, function=np.sin)

    p_in_nef = nengo.Probe(inp_nef)
    p_nef = nengo.Probe(outpt_nef, synapse=0.005)

with nengo_dl.Simulator(nef_net, seed=seed, dt=dt) as nef_sim:
    nef_sim.run_steps(n_steps_nef)
    t_nef = nef_sim.trange()
    x_nef = nef_sim.data[p_in_nef][:, 0]
    y_true = np.sin(x_nef ** 2)
    y_nef = nef_sim.data[p_nef][:, 0]

plt.figure(figsize=(10, 4))
plt.plot(t_nef, x_nef, label='x')
plt.plot(t_nef, y_true, label='True sin(x²)', lw=2)
plt.plot(t_nef, y_nef, label='NEF approximation (no gradient descent)')
plt.xlabel('time (s)')
plt.ylabel('value')
plt.title('NEF parameter optimization on a random temporal input')
plt.legend()
plt.tight_layout()
plt.show()

nef_mse = np.mean((y_true - y_nef) ** 2)
print(f'NEF approximation MSE: {nef_mse:.5f}')
print('NEF decoded weights were found analytically — zero gradient steps.')

### Combining NEF with gradient-based training

NEF gives a useful initialization, and `sim.fit()` can further refine trainable weights. In this small example the analytical NEF solution is already strong, so gradient updates may improve or worsen the result depending on the optimizer settings and the sampled signal.

In [ ]:
# Fine-tune the NEF solution with gradient descent.
# This is intentionally unguarded: if gradient training makes the model worse,
# the printed loss and plot should show that.

y_target = y_true.reshape(1, n_steps_nef, 1).astype(np.float32)

with nengo_dl.Simulator(nef_net, minibatch_size=1, seed=seed, dt=dt) as sim:
    sim.compile(
        optimizer=torch.optim.Adam(sim.trainable_params(), lr=1e-6),
        loss={p_nef: 'mse'},
    )

    result_nef = sim.evaluate(y={p_nef: y_target}, n_steps=n_steps_nef)
    print(f'MSE before gradient training (NEF only): {result_nef["loss"]:.5f}')

    sim.fit(y={p_nef: y_target}, n_steps=n_steps_nef, epochs=5)

    result_grad = sim.evaluate(y={p_nef: y_target}, n_steps=n_steps_nef)
    print(f'MSE after gradient training  (NEF + grad): {result_grad["loss"]:.5f}')

    sim.run_steps(n_steps_nef)
    y_grad = sim.data[p_nef][:, 0]

plt.figure(figsize=(10, 4))
plt.plot(t_nef, y_true, label='True sin(x²)', lw=2)
plt.plot(t_nef, y_nef, label='NEF only (analytical)')
plt.plot(t_nef, y_grad, label='NEF + gradient fine-tuning', linestyle=':')
plt.xlabel('time (s)')
plt.title('sin(x²): NEF initialization vs gradient training')
plt.legend()
plt.tight_layout()
plt.show()

print()
print('Key insight:')
print('  NEF:      fast, analytical, one layer at a time')
print('  Gradient: iterative, optimizes all trainable weights jointly')
print('  Combined: NEF can initialize a model, but gradient training is still an optimization problem')

## 7. Running on Neuromorphic Hardware

One of Nengo's strongest advantages over plain PyTorch is **backend portability**. A network defined once in Nengo can run on:

| Simulator | Backend | Use case |
|-----------|---------|----------|
| `nengo.Simulator` | CPU (NumPy) | Reference simulation, no dependencies |
| `nengo_dl.Simulator` | GPU (PyTorch) | Training and fast inference |
| `nengo_loihi.Simulator` | Intel Loihi chip | Ultra-low power spiking inference |
| `nengo_spinnaker.Simulator` | Manchester SpiNNaker | Massively parallel neuromorphic |

All backends consume the same `nengo.Network` object. You change only the simulator class. This makes it straightforward to:
1. Prototype in software with `nengo.Simulator`
2. Train with `nengo_dl.Simulator` (gradient-based, GPU)
3. Save weights with `sim.save_params()`
4. Load and deploy on neuromorphic hardware without rewriting the model.

Note: `nengo.Simulator` uses a different data convention — no batch dimension, shapes are `(n_steps, n_features)` instead of `(batch, n_steps, n_features)`.

In [ ]:
# Demonstrate backend portability: same nef_net on multiple simulators

print('The same nef_net network object can be simulated by:')
print('  nengo_dl.Simulator(nef_net)    <- this library (PyTorch, GPU)')
print('  nengo.Simulator(nef_net)       <- pure CPU (NumPy, no PyTorch)')
print('  nengo_loihi.Simulator(nef_net) <- Intel Loihi neuromorphic chip')
print('  nengo_spinnaker.Simulator(...) <- Manchester SpiNNaker platform')
print()

# The random input is generated by the Node inside nef_net, so no
# backend-specific data injection is required.
with nengo.Simulator(nef_net, seed=seed, dt=dt, progress_bar=False) as cpu_sim:
    cpu_sim.run_steps(n_steps_nef)
    cpu_t = cpu_sim.trange()
    cpu_in = cpu_sim.data[p_in_nef][:, 0]
    cpu_out = cpu_sim.data[p_nef][:, 0]

with nengo_dl.Simulator(nef_net, seed=seed, dt=dt) as dl_sim:
    dl_sim.run_steps(n_steps_nef)
    dl_t = dl_sim.trange()
    dl_in = dl_sim.data[p_in_nef][:, 0]
    dl_out = dl_sim.data[p_nef][:, 0]

cpu_true = np.sin(cpu_in ** 2)
dl_true = np.sin(dl_in ** 2)

print('Data shape conventions:')
print(f'  nengo.Simulator:    {cpu_out.shape}  -> (n_steps,) after selecting the probe dimension')
print(f'  nengo_dl.Simulator: {dl_out.shape}  -> (n_steps,) after selecting the probe dimension')
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(cpu_t, cpu_true, label='True sin(x²)', lw=2)
axes[0].plot(cpu_t, cpu_out, label='nengo.Simulator (CPU)')
axes[0].set_title('CPU backend (nengo.Simulator)')
axes[0].set_xlabel('time (s)')
axes[0].legend()

axes[1].plot(dl_t, dl_true, label='True sin(x²)', lw=2)
axes[1].plot(dl_t, dl_out, label='nengo_dl.Simulator (PyTorch)')
axes[1].set_title('PyTorch backend (nengo_dl.Simulator)')
axes[1].set_xlabel('time (s)')
axes[1].legend()

plt.suptitle('Same random-input network — two different backends', fontsize=12)
plt.tight_layout()
plt.show()

cpu_mse = np.mean((cpu_true - cpu_out) ** 2)
dl_mse = np.mean((dl_true - dl_out) ** 2)
print(f'CPU backend MSE:     {cpu_mse:.5f}')
print(f'PyTorch backend MSE: {dl_mse:.5f}')

print('Typical workflow:')
print('  1. Build network in Nengo')
print('  2. Train parameters with nengo_dl.Simulator (GPU-accelerated backprop)')
print('  3. Save weights: sim.save_params(path)')
print('  4. Load weights on neuromorphic hardware simulator')
print('  5. Deploy — same accuracy, dramatically lower power consumption')

## 8. Half-Domain Generalization

Now we can ask a more diagnostic question: if the model only sees half of the input range during optimization, does training the ensemble encoder parameters improve generalization on the held-out half?

The target below is intentionally similar to `sin(x²)` but not perfectly symmetric. We train on `x ∈ [-1, 0]` and evaluate on the full range `[-1, 1]`.

In [ ]:
# Data for a half-domain function approximation experiment.
rng = np.random.RandomState(seed)

n_train_half = 1024
n_eval_half = 401
minibatch_half = 64

# Smooth nonlinear target, close in spirit to sin(x²) but not even/symmetric.
def half_domain_fn(x):
    return np.sin(4 * x ** 2 + 0.5 * x)

x_train_half = rng.uniform(-1, 0, (n_train_half, 1, 1)).astype(np.float32)
y_train_half = half_domain_fn(x_train_half).astype(np.float32)

x_eval_half = np.linspace(-1, 1, n_eval_half, dtype=np.float32).reshape(-1, 1, 1)
y_eval_half = half_domain_fn(x_eval_half).astype(np.float32)

train_region = x_eval_half[:, 0, 0] <= 0
heldout_region = ~train_region

plt.figure(figsize=(9, 4))
plt.axvspan(-1, 0, color='tab:blue', alpha=0.08, label='training half')
plt.axvspan(0, 1, color='tab:orange', alpha=0.08, label='held-out half')
plt.plot(x_eval_half[:, 0, 0], y_eval_half[:, 0, 0], color='black', label='target')
plt.scatter(x_train_half[:80, 0, 0], y_train_half[:80, 0, 0], s=12, alpha=0.45, label='training samples')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('Training data covers only half of the input domain')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Build two comparable networks.
# Both solve NEF decoders using eval_points from only the training half.
# The second network then freezes connections and trains ensemble parameters.

half_eval_points = np.linspace(-1, 0, 600).reshape(-1, 1)


def build_half_domain_net(train_encoders=False):
    with nengo.Network(seed=seed) as net:
        if train_encoders:
            # Freeze connection decoders/transforms by default, then re-enable
            # training for Ensemble-owned parameters, then select only the encoder tensor in the optimizer.
            nengo_dl.configure_settings(trainable=False)
            net.config[nengo.Ensemble].trainable = True

        inp = nengo.Node(np.zeros(1))
        ens = nengo.Ensemble(
            80,
            dimensions=1,
            radius=1,
            eval_points=half_eval_points,
            neuron_type=nengo.LIFRate(),
        )
        out = nengo.Node(size_in=1)

        nengo.Connection(inp, ens, synapse=None)
        nengo.Connection(ens, out, function=lambda x: half_domain_fn(x[0]), synapse=None)
        probe = nengo.Probe(out, synapse=None)

    return net, inp, probe


nef_half_net, nef_half_inp, nef_half_probe = build_half_domain_net(train_encoders=False)
enc_half_net, enc_half_inp, enc_half_probe = build_half_domain_net(train_encoders=True)

In [ ]:
# Baseline: NEF solution trained only through the left-half eval_points.
with nengo_dl.Simulator(nef_half_net, minibatch_size=n_eval_half, seed=seed) as sim:
    sim.run_steps(1, data={nef_half_inp: x_eval_half})
    y_nef_half = sim.data[nef_half_probe][:, 0, 0]

# Gradient experiment: keep the NEF decoders fixed and train encoder weights only.
with nengo_dl.Simulator(enc_half_net, minibatch_size=minibatch_half, seed=seed) as sim:
    all_trainable = list(sim.trainable_params())
    encoder_params = [param for param in all_trainable if param.ndim == 2]
    print(f'Trainable parameter tensors exposed by the simulator: {len(all_trainable)}')
    print(f'Encoder tensors used by the optimizer: {len(encoder_params)}')
    sim.compile(
        optimizer=torch.optim.Adam(encoder_params, lr=2e-3),
        loss={enc_half_probe: 'mse'},
    )

    before = sim.evaluate(
        x={enc_half_inp: x_train_half},
        y={enc_half_probe: y_train_half},
        n_steps=1,
    )
    print(f'Training-half MSE before encoder-weight training: {before["loss"]:.5f}')

    sim.fit(
        x={enc_half_inp: x_train_half},
        y={enc_half_probe: y_train_half},
        n_steps=1,
        epochs=25,
    )

    after = sim.evaluate(
        x={enc_half_inp: x_train_half},
        y={enc_half_probe: y_train_half},
        n_steps=1,
    )
    print(f'Training-half MSE after encoder-weight training:  {after["loss"]:.5f}')
    trained_encoder_weights = sim.get_weights()

# Rebuild with a larger minibatch for plotting all evaluation points at once.
with nengo_dl.Simulator(enc_half_net, minibatch_size=n_eval_half, seed=seed) as sim:
    sim.set_weights(trained_encoder_weights)
    sim.run_steps(1, data={enc_half_inp: x_eval_half})
    y_enc_half = sim.data[enc_half_probe][:, 0, 0]


def region_mse(y_pred, mask):
    target = y_eval_half[:, 0, 0]
    return np.mean((target[mask] - y_pred[mask]) ** 2)

metrics = {
    'NEF train half': region_mse(y_nef_half, train_region),
    'NEF held-out half': region_mse(y_nef_half, heldout_region),
    'Encoder weights train half': region_mse(y_enc_half, train_region),
    'Encoder weights held-out half': region_mse(y_enc_half, heldout_region),
}

for name, value in metrics.items():
    print(f'{name:24s}: {value:.5f}')

In [ ]:
# Visualize interpolation on the trained half and extrapolation on the held-out half.
x_axis_half = x_eval_half[:, 0, 0]
y_axis_half = y_eval_half[:, 0, 0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].axvspan(-1, 0, color='tab:blue', alpha=0.08)
axes[0].axvspan(0, 1, color='tab:orange', alpha=0.08)
axes[0].plot(x_axis_half, y_axis_half, color='black', lw=2, label='target')
axes[0].plot(x_axis_half, y_nef_half, label='NEF decoders only')
axes[0].plot(x_axis_half, y_enc_half, label='NEF + trained encoder weights', linestyle=':')
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].set_title('Function approximation across the full domain')
axes[0].legend()

bar_names = ['NEF train', 'NEF held out', 'Encoder train', 'Encoder held out']
axes[1].set_ylabel('MSE')
axes[1].set_title('Error split by seen vs held-out input range')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print('Question to inspect: does reducing training error on [-1, 0] also improve the held-out half?')

## Summary

This tutorial covered the key concepts for PyTorch users coming to NengoDL:

| Topic | PyTorch equivalent | Nengo/NengoDL |
|-------|-------------------|---------------|
| Layer | `nn.Linear` + activation | `nengo.Ensemble` + `neuron_type` |
| Weight matrix | `nn.Linear.weight` | `nengo.Connection` transform |
| Output capture | return tensor | `nengo.Probe` |
| Forward pass | `model(x)` | `sim.run_steps(n, data={...})` |
| Data shape | `(batch, features)` | `(batch, n_steps, features)` |
| Rate neurons | ReLU, sigmoid | `nengo.RectifiedLinear`, `nengo.LIFRate` |
| Spiking neurons | — | `nengo.LIF`, `nengo.SpikingRectifiedLinear` |
| Custom module | any `nn.Module` | `nengo_dl.TorchNode(module, ...)` |
| Gradient training | `optimizer.step()` | `sim.compile()` + `sim.fit()` |
| Analytical optimization | — | NEF via `Connection(function=...)` |
| Hardware deployment | CUDA / CPU | + Loihi, SpiNNaker, etc. |

**Next steps:**
- See `spiking-mnist.ipynb` for a full spiking network trained on MNIST
- See `from-nengo.ipynb` for details on Nengo's native population coding
- See the [REFERENCE.md](../../REFERENCE.md) for full API documentation